# Music Library Tools — Example Notebook

This notebook demonstrates the four tools built on top of this repository:

| Tool | Module | What it does |
|---|---|---|
| Library Diff | `library_diff` | Find albums present in one library but not another; move/copy the surplus |
| Incomplete Albums | `incomplete_albums` | Detect albums with gaps in track numbering (local) or fewer tracks than MusicBrainz expects (online) |
| Streaming Finder | `fetch_incomplete_on_streaming` | Search the incomplete-album list across Deezer, Spotify, Apple Music, Tidal, Qobuz, Amazon Music |
| Empty Dir Cleaner | `remove_empty_dirs` | Walk a path tree and delete every empty directory |

All four modules also ship with a `main()` so they can be run as standalone CLI scripts.

---
## Setup

In [ ]:
# Install required packages (skip if already installed)
# !pip install mutagen requests

In [ ]:
import json
from pathlib import Path

# Shared library scanner — used by all tools
from music_library import scan_albums_in_dir, scan_album_dirs, AUDIO_EXTENSIONS

# Tool modules
import library_diff          as ld
import incomplete_albums     as ia
import fetch_incomplete_on_streaming as fis
import remove_empty_dirs     as red

---
## Part 0 — Shared music library scanner (`music_library`)

`scan_albums_in_dir` is the low-level generator that all tools share.
Use it whenever you just need to iterate over albums without loading metadata.

In [ ]:
# ── Adjust this to your actual library path ──
MUSIC_DIR = Path("E:/Music")

# Generator — does not load metadata, just yields (artist_name, album_name, album_dir)
all_albums = list(scan_albums_in_dir(MUSIC_DIR))
print(f"Found {len(all_albums)} albums")
print("First 5:")
for artist, album, path in all_albums[:5]:
    print(f"  {artist} / {album}")

---
## Part 1 — Library Diff (`library_diff`)

Given a **source** library (the larger / surplus one) and a **reference** library
(the "main" one you want to keep in sync with), `find_surplus` returns album
directories that are in source but absent from reference.

Matching is **case-insensitive** on both artist and album name.

In [ ]:
# ── Adjust paths ──
SOURCE_DIR    = Path("E:/MusicSurplus")
REFERENCE_DIR = Path("E:/Music")
SURPLUS_DEST  = Path("E:/Surplus")     # where to move/copy surplus albums

# 1 — Index both libraries (case-insensitive normalised dict)
source_idx    = ld.scan_library(SOURCE_DIR)
reference_idx = ld.scan_library(REFERENCE_DIR)

# 2 — Find what is in source but not in reference
surplus = ld.find_surplus(source_idx, reference_idx)
print(f"\n{len(surplus)} surplus album(s) found:")
for album_path in surplus[:10]:
    print(f"  {album_path.relative_to(SOURCE_DIR)}")

In [ ]:
# 3 — Inspect the report dict without touching the filesystem
report = ld.build_diff_report(
    surplus, SOURCE_DIR, REFERENCE_DIR,
    destination=SURPLUS_DEST,
    move=True,
    dry_run=True,
)
print(json.dumps(report, indent=2))

In [ ]:
# 4 — DRY RUN: see what would be moved, nothing is changed
ok, fail = ld.transfer_surplus(
    surplus_albums=surplus,
    source_root=SOURCE_DIR,
    destination=SURPLUS_DEST,
    move=True,
    dry_run=True,     # ← set to False to actually move
)
print(f"Would move: {ok}  failures: {fail}")

---
## Part 2 — Incomplete Albums (`incomplete_albums`)

Two checks are combined:

* **Local** — reads `tracknumber` tags (or falls back to `NN - title` filename parsing) and
  detects gaps in the sequence or a count below the `totaltracks` tag.
* **Online** (optional) — queries MusicBrainz for the expected track count.

The result is a list of plain dicts — easy to inspect in a notebook or pipe to the next step.

In [ ]:
# ── Adjust path ──
SURPLUS_DIR = Path("E:/Surplus")   # or any library directory

# Local-only check (fast, no network)
incomplete = ia.scan_library_for_incomplete(
    SURPLUS_DIR,
    online=False,   # set to True to also query MusicBrainz (slow!)
    limit=50,       # remove limit for full scan
)

print(f"\n{len(incomplete)} incomplete album(s) found")
for entry in incomplete[:5]:
    print(f"  {entry['album_artist']} / {entry['album']}")
    print(f"    ↳ {entry['reason']}")
    print(f"    local_tracks={entry['local_tracks']}  mb_total={entry['mb_total']}")

In [ ]:
# Check a single album directory directly
album_dir = Path("E:/Surplus/Radiohead/OK Computer")
ok, reason, tracks, declared = ia.check_local_completeness(album_dir)
print(f"Complete: {ok}")
print(f"Reason  : {reason}")
print(f"Tracks  : {sorted(tracks)}")
print(f"Declared: {declared}")

In [ ]:
# Save for the next step
OUTPUT_JSON = Path("incomplete_albums.json")
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(incomplete, f, indent=2, ensure_ascii=False)
print(f"Saved {len(incomplete)} entries to {OUTPUT_JSON}")

---
## Part 3 — Find Incomplete Albums on Streaming Services (`fetch_incomplete_on_streaming`)

Consumes the list from Part 2 and queries Deezer (free API), Spotify, Apple Music,
Tidal, Qobuz, and Amazon Music.

You can pass your own `StreamingSearcher` instance (e.g. one with Spotify credentials
loaded) via the `searcher=` argument, or leave it `None` to auto-create one from
`streaming_config.json`.

In [ ]:
from find_albums_on_streaming import StreamingSearcher

# Optional: provide explicit Spotify credentials
# searcher = StreamingSearcher(
#     spotify_client_id="YOUR_ID",
#     spotify_client_secret="YOUR_SECRET",
# )

# Auto-load from streaming_config.json (or no Spotify if file absent)
searcher = StreamingSearcher()

In [ ]:
# Load the incomplete list (from file or directly from the list produced above)
albums_to_search = fis.load_incomplete_albums(OUTPUT_JSON)
# -- or use the in-memory list directly: --
# albums_to_search = incomplete

# Search (limit= is useful during development)
results = fis.search_albums_on_streaming(
    albums_to_search,
    searcher=searcher,
    limit=5,       # remove for full run
)

# Quick summary
for r in results:
    found = [svc for svc, url in r["streaming"].items() if url]
    print(f"{r['album_artist']} / {r['album']}  → {found or 'not found'}")

In [ ]:
# Write reports
fis.write_text_report(results, Path("streaming_incomplete.txt"))
fis.write_json_report(results, Path("streaming_incomplete.json"))
print("Reports written.")

---
## Part 4 — Remove Empty Directories (`remove_empty_dirs`)

After moving surplus albums out of a directory the parent folders may become
empty. `remove_empty_dirs` walks the tree **bottom-up** so that removing a
leaf can expose its parent for removal in the same pass.

In [ ]:
CLEAN_DIR = Path("E:/MusicSurplus")  # the directory to clean up

# Dry run first — see what would be deleted
count = red.remove_empty_dirs(CLEAN_DIR, dry_run=True, keep_root=True)
print(f"Would remove {count} empty director{'y' if count == 1 else 'ies'}")

In [ ]:
# Uncomment to actually delete
# count = red.remove_empty_dirs(CLEAN_DIR, dry_run=False, keep_root=True)
# print(f"Removed {count} empty director{'y' if count == 1 else 'ies'}")

---
## Part 5 — Full Pipeline

String all four steps together for a complete workflow:

1. Diff two libraries → isolate surplus
2. Move surplus to a staging area
3. Find incomplete albums in the staging area
4. Look them up on streaming services
5. Clean up empty directories left behind

In [ ]:
SOURCE    = Path("E:/MusicSurplus")
REFERENCE = Path("E:/Music")
STAGING   = Path("E:/Staging")

# ── Step 1 & 2: diff + move surplus ──────────────────────────────────────
surplus = ld.find_surplus(ld.scan_library(SOURCE), ld.scan_library(REFERENCE))
print(f"Surplus: {len(surplus)} albums")

ok, fail = ld.transfer_surplus(surplus, SOURCE, STAGING, move=True, dry_run=True)
print(f"Transfer (dry-run): {ok} ok / {fail} failed")

# ── Step 3: find incomplete albums in staging area ────────────────────────
incomplete = ia.scan_library_for_incomplete(STAGING, online=False)
print(f"Incomplete: {len(incomplete)} albums")

# ── Step 4: search streaming services ────────────────────────────────────
results = fis.search_albums_on_streaming(incomplete[:3])   # limit for demo
fis.write_text_report(results, Path("pipeline_streaming.txt"))
print("Streaming report written.")

# ── Step 5: clean empty dirs ──────────────────────────────────────────────
n = red.remove_empty_dirs(SOURCE, dry_run=True, keep_root=True)
print(f"Would remove {n} empty dirs from source")

---
## Playlist Matcher (existing tool)

The original `playlist_matcher.py` is unchanged and works well alongside these tools.
The pattern from the existing `playlists.ipynb`:

In [ ]:
import playlist_matcher as pm

matcher = pm.PlaylistMatcher(
    playlist_path=Path("playlists/MyPlaylist.txt"),
    music_dir="E:/Music",
    output_path=Path("E:/Playlists/MyPlaylist.m3u8"),
    log_path=Path("unmatched.log"),
    path_format="artist_album",
)

# Build (or load from cache) the metadata index
matcher.build_library_cache()

# Run
lines = matcher.read_old_playlist()
matched, unmatched = matcher.find_matches(lines)
matcher.write_new_playlist(matched)
matcher.write_log(matched, unmatched)

In [ ]:
from album_overlap import find_overlapping_albums, write_text_report

results = find_overlapping_albums(
    Path("E:/Music"),
    threshold=0.4,        # report pairs with ≥ 40% overlap
    duration_bucket=10,   # looser duration matching
)

for r in results:
    print(f"[{r['overlap']:.0%}] {r['album_artist']} — {r['album_a']!r} ↔ {r['album_b']!r}")

write_text_report(results, Path("overlaps.txt"))